[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C27_Model_Compression_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与压缩方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零实现**每个压缩算法，再与**全精度参考**对拍。

这个 notebook 做四件事：① 确认环境；② 用一个最小例子体会「**压缩省的是显存/带宽，不是算力**」；③ 把「压缩 = 带约束的近似」这个统一框架跑一遍；④ 立下全课的纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画精度-压缩率曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 压缩省的是显存与带宽

用一个真实规模的线性层算笔账：把权重从 **fp16（2 字节）** 压到 **int4（0.5 字节）**，**显存** 和 **每次前向要从显存搬运的字节数** 各省多少？

这解释了为什么访存受限的 LLM 推理里，weight-only 量化即便要反量化也能加速——**省的是带宽**。

In [ ]:
def layer_bytes(d_in, d_out, bytes_per_weight):
    return d_in * d_out * bytes_per_weight

# 一个 7B 模型典型的 MLP 投影层尺寸量级
d_in, d_out = 4096, 11008
for name, b in [('fp16', 2.0), ('int8', 1.0), ('int4', 0.5)]:
    mb = layer_bytes(d_in, d_out, b) / 1e6
    print(f'{name:>5s}: 单层权重 {mb:8.1f} MB  (相对 fp16 压缩 {2.0/b:.0f}x)')
fp16_mb = layer_bytes(d_in, d_out, 2.0) / 1e6
int4_mb = layer_bytes(d_in, d_out, 0.5) / 1e6
assert abs(fp16_mb / int4_mb - 4.0) < 1e-6
print('\nint4 把权重显存与每次前向的权重搬运量都降到 1/4 —— 访存受限推理直接受益 ✅')

**关键结论**：量化最直接的收益是**显存占用**和**带宽（要搬的字节）**按 bit 数成比例下降。

逐 token 生成的 LLM 推理大多访存受限：瓶颈在搬几十 GB 权重、不在算乘加。所以把权重从 16 bit 压到 4 bit，即便推理时要反量化（多算一点），整体仍更快——这就是 weight-only 量化（GPTQ/AWQ）的底层逻辑。

## 3 · 统一框架：压缩 = 带约束的近似

本课所有压缩都在解同一个问题：找一个**便宜**的 $\hat W$ 近似原 $W$，使**层输出**尽量不变：

$$\min_{\hat W \in \mathcal{C}} \; \| W x - \hat W x \|^2$$

约束 $\mathcal{C}$ 不同：量化要求落在网格上、剪枝要求很多 0。先用最简单的量化约束跑一遍这个框架。

In [ ]:
rng = np.random.default_rng(0)
d_in, d_out, n = 64, 32, 128
W = rng.standard_normal((d_out, d_in)) * 0.1     # 原权重
X = rng.standard_normal((d_in, n))               # 校准激活

def quantize_per_tensor_int8(W):
    s = np.abs(W).max() / 127.0                   # absmax 对称 scale
    q = np.round(W / s).clip(-127, 127)           # 量化到网格
    return q, s

q, s = quantize_per_tensor_int8(W)
W_hat = s * q                                     # 反量化
out_err = np.linalg.norm(W @ X - W_hat @ X)       # 我们真正在意的：输出误差
wt_err  = np.linalg.norm(W - W_hat)               # 权重误差（顺带看看）
print(f'int8 量化: 输出误差 ‖Wx-Ŵx‖={out_err:.4f}, 权重误差 ‖W-Ŵ‖={wt_err:.4f}')
# 量化后输出应当很接近（int8 近乎无损）
rel = out_err / np.linalg.norm(W @ X)
assert rel < 0.02, rel
print(f'相对输出误差 {rel:.3%} —— int8 近乎无损。后续模块就是在更狠的约束下把这个误差压住 ✅')

**注意**优化目标是 $\|Wx-\hat Wx\|$（**输出**接近）而非 $\|W-\hat W\|$（权重接近）。

这个区别正是 GPTQ（用激活的 Hessian 加权误差）和 AWQ（按激活幅度保护通道）比朴素就近舍入更准的根本原因——它们把**激活 $x$** 纳入了考量。模块 02 会从零复现这一点。

## 4 · 立纪律：对拍（differential testing）

本课每个压缩算法都要和**全精度参考**比对。先把这个工作流跑通，并验证一个会反复用到的事实：**量化误差不是玄学**——均匀量化的单元素误差近似服从 $[-s/2, s/2]$ 均匀分布，期望平方误差约 $s^2/12$。

In [ ]:
def quantize_dequantize(x, s, qmin, qmax):
    q = np.round(x / s).clip(qmin, qmax)
    return s * q

# 取一段不会被截断的数据，验证量化误差的统计规律
s = 0.05
x = rng.uniform(-2, 2, size=200_000)             # 远离截断边界
xq = quantize_dequantize(x, s, qmin=-1000, qmax=1000)
err = xq - x
print(f'误差范围: [{err.min():.4f}, {err.max():.4f}]  (理论 ±s/2 = ±{s/2:.4f})')
mse = (err ** 2).mean()
print(f'实测 MSE = {mse:.6e},  理论 s^2/12 = {s**2/12:.6e}')
assert err.max() <= s/2 + 1e-9 and err.min() >= -s/2 - 1e-9
assert abs(mse - s**2/12) / (s**2/12) < 0.05    # 5% 以内
print('✅ 量化误差服从均匀分布、MSE≈s²/12 —— 量化噪声是可预测、可计算的，不是玄学')

## 5 · 各种精度格式：bit 越少，能表示的「格子」越少

量化的本质是减少可表示的离散级别数。直观感受一下不同 bit 数对应多少个「格子」，以及为什么 int4 这么少的格子需要精细的分组与误差补偿才能保住精度。

In [ ]:
formats = [
    ('fp16', 'float',  65504,   '~万级有效格子（非均匀）'),
    ('int8', 'int',    256,     '256 级（均匀）'),
    ('int4', 'int',    16,      '仅 16 级！需 group-wise + 补偿'),
    ('fp8 E4M3', 'float', 448,  '~256 个非均匀格子, 范围±448'),
]
print(f"{'格式':<10}{'类型':<7}{'级数/上界':>10}  说明")
for name, kind, levels, note in formats:
    print(f'{name:<10}{kind:<7}{levels:>10}  {note}')
# int4 只有 16 个级别：这就是为什么它最难、最需要本课模块 01/02 的技巧
n_levels_int4 = 2 ** 4
assert n_levels_int4 == 16
print('\n✅ int4 只有 16 个离散级别 —— 模块 01(分组) 与 02(GPTQ/AWQ 补偿) 就是为了让这 16 格够用')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成小函数，后面每个模块都用它判定「我的压缩结果 vs 全精度参考」。它就是本课所有 `assert` 背后的统一裁判。

In [ ]:
def report(name, got, ref, atol=1e-8):
    '''对拍：压缩/重建结果 vs 全精度参考。打印最大误差与是否一致。'''
    got = np.asarray(got, dtype=float); ref = np.asarray(ref, dtype=float)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    ok = max_err <= atol
    print(f'[{name:<28}] max|err|={max_err:.3e}  ok={ok}')
    return max_err

def rel_output_error(W, W_hat, X):
    '''层输出的相对误差 ‖Wx-Ŵx‖/‖Wx‖ —— 压缩质量的核心度量。'''
    return np.linalg.norm(W @ X - W_hat @ X) / np.linalg.norm(W @ X)

# 演示：int8 vs int4 per-tensor 的输出误差差距
def quant_pt(W, bits):
    qmax = 2 ** (bits - 1) - 1
    s = np.abs(W).max() / qmax
    return s * np.round(W / s).clip(-qmax, qmax)

e8 = rel_output_error(W, quant_pt(W, 8), X)
e4 = rel_output_error(W, quant_pt(W, 4), X)
print(f'per-tensor int8 相对输出误差 = {e8:.3%}')
print(f'per-tensor int4 相对输出误差 = {e4:.3%}  <- 明显更大，模块 01/02 来拯救它')
assert e4 > e8, 'int4 误差应大于 int8'
print('\n✅ 工作流就绪：写压缩 -> 对拍全精度 -> assert 兜底。这是全课的契约。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个量化/补偿/蒸馏/剪枝算法，都会用全精度参考对拍；数值正确则逻辑可迁移到 bitsandbytes / AutoGPTQ / TransformerEngine 等真实框架。

**接下来五个模块**：01 整数量化 → 02 GPTQ/AWQ → 03 fp8 训练 → 04 知识蒸馏 → 05 剪枝与稀疏。

下一站：**模块 01 · 整数量化**。